1. Load raw dataset

In [0]:
df_chikungunya_spatial_raw = spark.table("raw_chikungunya_espacial")

display(df_chikungunya_spatial_raw)
print(df_chikungunya_spatial_raw.columns)

2. Clean and standardize

In [0]:
from pyspark.sql.functions import col, lit, trim, upper, row_number
from pyspark.sql.window import Window

df_chikungunya_spatial_clean = (
    df_chikungunya_spatial_raw
    .withColumn("row_id", row_number().over(Window.orderBy(lit(1))))
    .filter(col("row_id") > 3)
    .drop("row_id")
)

df_chikungunya_spatial = (
    df_chikungunya_spatial_clean
    .select(
        col("_c1").cast("int").alias("idade"),
        col("_c2").alias("sexo"),
        col("_c3").alias("bairro")
    )
    .withColumn("doenca", lit("CHIKUNGUNYA"))
    .withColumn("bairro", trim(upper(col("bairro"))))
)

3. Save

In [0]:
df_chikungunya_spatial.write.mode("overwrite").option("overwriteSchema", "true").saveAsTable("fato_chikungunya_espacial")

4. Validate

In [0]:
df_fato_chikungunya_spatial = spark.table("fato_chikungunya_espacial")

display(df_fato_chikungunya_spatial)
print(df_fato_chikungunya_spatial.count())
print(df_fato_chikungunya_spatial.columns)